In [1]:
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH, SILVER_DB, GOLD_DB

# ETL Silver → Gold (Esquema Estrella)
Construye el Star Schema del Data Mart:
- `dim_date`: generada con pandas.date_range (2005-2007)
- `dim_film`: film + language + category
- `dim_customer`: customer + address + city + country
- `dim_staff`: staff
- `dim_store`: store + staff (gerente) + address + city + country
- `dim_actor`: actor
- `fact_rental`: rental + inventory + payment (tabla central de hechos)

In [2]:
def cargar_a_gold(nombre_tabla, query):
    print(f'Construyendo gold.{nombre_tabla}...')
    rows, cols = CH.execute(query, with_column_types=True)
    df = pd.DataFrame(rows, columns=[c[0] for c in cols])

    # Convertir NaT → None
    for col in df.columns:
        try:
            mask = df[col].isna()
            df[col] = df[col].astype(object)
            df.loc[mask, col] = None
        except:
            pass

    df['_valid_from'] = datetime.now()
    df = df.where(pd.notnull(df), None)

    CH.execute(f'TRUNCATE TABLE {GOLD_DB}.{nombre_tabla}')
    CH.execute(f'INSERT INTO {GOLD_DB}.{nombre_tabla} VALUES', df.to_dict('records'))
    print(f'  ✓ {len(df):,} filas cargadas en gold.{nombre_tabla}')

## 1. Dimensión Fecha
Generada con pandas para el rango 2005-2007 (rango de datos de Sakila).

In [3]:
def cargar_dim_date():
    print('Generando gold.dim_date...')
    dates = pd.date_range('2005-01-01', '2007-12-31', freq='D')

    df = pd.DataFrame({
        'date_key':     dates.strftime('%Y%m%d').astype(int),
        'full_date':    dates.date,
        'year':         dates.year.astype(int),
        'quarter':      dates.quarter.astype(int),
        'quarter_name': ('Q' + dates.quarter.astype(str)),
        'month':        dates.month.astype(int),
        'month_name':   dates.strftime('%B'),
        'week':         dates.isocalendar().week.astype(int),
        'day':          dates.day.astype(int),
        'day_name':     dates.strftime('%A'),
        'is_weekend':   (dates.weekday >= 5).astype(int),
    })

    CH.execute(f'TRUNCATE TABLE {GOLD_DB}.dim_date')
    CH.execute(
        f"INSERT INTO {GOLD_DB}.dim_date ({','.join(df.columns)}) VALUES",
        df.to_dict('records')
    )
    print(f'  ✓ {len(df):,} fechas cargadas en gold.dim_date')

cargar_dim_date()

Generando gold.dim_date...
  ✓ 1,095 fechas cargadas en gold.dim_date


## 2. Dimensiones

In [4]:
dims = {
    'dim_film': f"""
        SELECT
            f.film_id         AS film_key,
            f.title,
            f.description,
            f.release_year,
            l.name            AS language,
            cat.name          AS category,
            f.rating,
            f.rental_duration,
            f.rental_rate,
            f.length          AS length_minutes,
            f.replacement_cost
        FROM {SILVER_DB}.stg_film f
        LEFT JOIN {SILVER_DB}.stg_language      l   ON f.language_id  = l.language_id
        LEFT JOIN {SILVER_DB}.stg_film_category fc  ON f.film_id      = fc.film_id
        LEFT JOIN {SILVER_DB}.stg_category      cat ON fc.category_id = cat.category_id
    """,
    'dim_customer': f"""
        SELECT
            c.customer_id AS customer_key,
            c.full_name,
            c.email,
            a.address,
            a.district,
            ci.city,
            co.country,
            a.postal_code,
            c.active,
            c.store_id
        FROM {SILVER_DB}.stg_customer c
        LEFT JOIN {SILVER_DB}.stg_address a  ON c.address_id  = a.address_id
        LEFT JOIN {SILVER_DB}.stg_city    ci ON a.city_id     = ci.city_id
        LEFT JOIN {SILVER_DB}.stg_country co ON ci.country_id = co.country_id
    """,
    'dim_staff': f"""
        SELECT
            staff_id AS staff_key,
            full_name,
            email,
            store_id,
            username,
            active
        FROM {SILVER_DB}.stg_staff
    """,
    'dim_store': f"""
        SELECT
            st.store_id   AS store_key,
            sf.full_name  AS manager_name,
            a.address,
            ci.city,
            co.country
        FROM {SILVER_DB}.stg_store st
        LEFT JOIN {SILVER_DB}.stg_staff   sf ON st.manager_staff_id = sf.staff_id
        LEFT JOIN {SILVER_DB}.stg_address a  ON st.address_id       = a.address_id
        LEFT JOIN {SILVER_DB}.stg_city    ci ON a.city_id           = ci.city_id
        LEFT JOIN {SILVER_DB}.stg_country co ON ci.country_id       = co.country_id
    """,
    'dim_actor': f"""
        SELECT
            actor_id AS actor_key,
            full_name
        FROM {SILVER_DB}.stg_actor
    """,
}

print('=== Cargando Dimensiones ===')
for tabla, query in dims.items():
    cargar_a_gold(tabla, query)

=== Cargando Dimensiones ===
Construyendo gold.dim_film...
  ✓ 1,000 filas cargadas en gold.dim_film
Construyendo gold.dim_customer...
  ✓ 599 filas cargadas en gold.dim_customer
Construyendo gold.dim_staff...
  ✓ 2 filas cargadas en gold.dim_staff
Construyendo gold.dim_store...
  ✓ 2 filas cargadas en gold.dim_store
Construyendo gold.dim_actor...
  ✓ 200 filas cargadas en gold.dim_actor


## 3. Tabla de Hechos — fact_rental
Granularidad: una fila por renta.  
JOINs: rental + inventory (film, store) + payment (monto).

In [5]:
fact_query = f"""
    SELECT
        r.rental_id                                                AS rental_id,
        toInt32(formatDateTime(r.rental_date, '%Y%m%d'))          AS date_key,
        i.film_id                                                  AS film_key,
        r.customer_id                                              AS customer_key,
        r.staff_id                                                 AS staff_key,
        i.store_id                                                 AS store_key,
        r.inventory_id                                             AS inventory_id,
        ifNull(p.amount, 0)                                        AS amount,
        dateDiff('day', r.rental_date,
                 ifNull(r.return_date, now()))                     AS rental_duration_days,
        r.rental_date                                              AS rental_date,
        r.return_date                                              AS return_date,
        p.payment_date                                             AS payment_date
    FROM {SILVER_DB}.stg_rental r
    LEFT JOIN {SILVER_DB}.stg_inventory i ON r.inventory_id = i.inventory_id
    LEFT JOIN {SILVER_DB}.stg_payment   p ON r.rental_id   = p.rental_id
"""

print('Construyendo gold.fact_rental...')
rows, cols = CH.execute(fact_query, with_column_types=True)
df = pd.DataFrame(rows, columns=[c[0] for c in cols])

# Convertir NaT → None en columnas de fecha
for col in ['rental_date', 'return_date', 'payment_date']:
    mask = df[col].isna()
    df[col] = df[col].astype(object)
    df.loc[mask, col] = None

df = df.where(pd.notnull(df), None)
CH.execute(f'TRUNCATE TABLE {GOLD_DB}.fact_rental')
CH.execute(f'INSERT INTO {GOLD_DB}.fact_rental VALUES', df.to_dict('records'))
print(f'  ✓ {len(df):,} hechos cargados en gold.fact_rental')

Construyendo gold.fact_rental...
  ✓ 16,044 hechos cargados en gold.fact_rental


## 4. Verificación Final Gold

In [6]:
print('=== VERIFICACIÓN GOLD ===')
tablas_gold = [
    ('dim_film',     1000),
    ('dim_customer', 599),
    ('dim_staff',    2),
    ('dim_store',    2),
    ('dim_actor',    200),
    ('dim_date',     1095),
    ('fact_rental',  16044),
]
for tabla, esperado in tablas_gold:
    cnt = CH.execute(f'SELECT count() FROM gold.{tabla}')[0][0]
    status = '✓' if cnt == esperado else '✗ ALERTA'
    print(f'  {status} gold.{tabla}: {cnt:,} filas (esperado: {esperado:,})')

=== VERIFICACIÓN GOLD ===
  ✓ gold.dim_film: 1,000 filas (esperado: 1,000)
  ✓ gold.dim_customer: 599 filas (esperado: 599)
  ✓ gold.dim_staff: 2 filas (esperado: 2)
  ✓ gold.dim_store: 2 filas (esperado: 2)
  ✓ gold.dim_actor: 200 filas (esperado: 200)
  ✓ gold.dim_date: 1,095 filas (esperado: 1,095)
  ✓ gold.fact_rental: 16,044 filas (esperado: 16,044)
